# 知识点闭环总结：从 RNN 到 Transformer（246 → 263）

> 这一份是把第 **246~263** 共 18 章串成一条主线的复习笔记。核心叙事只有一句话：
>
> **如何建模序列 → 如何记得更久 → 如何处理「序列对」→ 如何打破信息瓶颈 → 如何彻底抛弃循环。**
>
> 每一章都在解决上一章暴露出来的具体缺陷。读这份总结时，请始终带着两个贯穿全程的问题：
>
> 1. **记忆 vs 并行**：怎样既记住远处的信息，又能高效并行计算？
> 2. **query / key / value**：后半段所有模型都在用这套统一语言，注意它在不同模型里「谁当 query、谁当 key/value」。

## 0. 全景地图（一张表看懂 18 章）

| 阶段 | 章节 | 主题 | 解决的核心问题 |
|------|------|------|----------------|
| **一、序列基础** | 246 序列模型 | 自回归、马尔可夫假设 | 序列数据怎么建模？ |
| | 247 文本预处理 | 词元化、词表 Vocab | 文本怎么变成数字？ |
| | 248 语言模型 | n-gram、困惑度 | 怎么衡量「一句话像不像话」？ |
| **二、循环网络** | 249/250 RNN | 隐状态、BPTT | 怎么让网络拥有「记忆」？ |
| | 251 GRU | 重置门/更新门 | 怎么缓解长程梯度消失？ |
| | 252 LSTM | 输入/遗忘/输出门 + 记忆元 | 怎么更精细地控制记忆？ |
| | 253 深层 RNN | 多层堆叠 | 怎么增强表达能力？ |
| | 254 双向 RNN | 前向+后向 | 怎么利用「未来」的上下文？ |
| **三、序列到序列** | 255 机器翻译数据集 | 预处理、`<bos>/<eos>/<pad>` | 翻译数据怎么准备？ |
| | 256 编码器-解码器 | Encoder-Decoder 架构 | 输入输出都是序列怎么办？ |
| | 257 seq2seq | RNN 编码器+解码器、BLEU | 怎么端到端做翻译？ |
| | 258 束搜索 | 贪心/穷举/束搜索 | 推理时怎么搜出好序列？ |
| **四、注意力→Transformer** | 259 注意力机制 | query/key/value、核回归 | 怎么「按需关注」输入？ |
| | 260 注意力分数 | 加性 / 缩放点积 | 注意力权重怎么算？ |
| | 261 注意力 seq2seq | Bahdanau 注意力 | 怎么打破固定上下文瓶颈？ |
| | 262 自注意力 | self-attention + 位置编码 | 怎么让序列自己看自己？ |
| | 263 Transformer | 多头+掩码+残差+LayerNorm | 怎么纯靠注意力搭模型？ |

## 一、序列基础（246–248）：让数据「可建模、可输入、可评价」

### 246 序列模型
序列数据的特点是**有先后顺序、前后相关**。建模目标是给定历史预测下一步，即联合概率按时间分解：

$$p(x_1, \dots, x_T) = \prod_{t=1}^{T} p(x_t \mid x_1, \dots, x_{t-1})$$

- **字符定义**：$x_t$ 是第 $t$ 个时间步的观测，$T$ 是序列长度，$p(x_t \mid \cdots)$ 是给定历史的条件概率。
- **公式描述**：把「整句话的概率」拆成「每个词在它前文条件下出现的概率」的连乘——这就是**自回归（autoregressive）**思想。
- **马尔可夫假设**：为简化，假设当前只依赖最近 $\tau$ 步（$p(x_t \mid x_{t-\tau}, \dots, x_{t-1})$），这样输入长度固定，可直接用 MLP。

### 247 文本预处理
把文本变成模型能吃的数字，流程：**读取 → 词元化(tokenize) → 构建词表(Vocab) → 映射成索引**。词表把每个词元映射到整数 id，并保留特殊词元（`<unk>` 未知词等）。

### 248 语言模型
语言模型就是估计上面那个 $p(x_1,\dots,x_T)$。早期用 **n-gram**（基于计数 + 马尔可夫假设）。评价指标是**困惑度（perplexity）**：

$$\mathrm{PPL} = \exp\!\left(-\frac{1}{n}\sum_{t=1}^{n} \log p(x_t \mid x_{t-1}, \dots, x_1)\right)$$

- **字符定义**：$n$ 是总词元数，$p(x_t \mid \cdots)$ 是模型对真实下一个词给出的概率。
- **公式描述**：困惑度是「平均交叉熵损失」的指数。**直观理解**：模型在每一步「纠结」于多少个候选词。PPL=1 完美，PPL=词表大小 相当于瞎猜。越小越好。

## 二、循环神经网络（249–254）：给网络装上「记忆」

### 249/250 RNN —— 用隐状态承载记忆
RNN 引入一个随时间传递的**隐状态 $\mathbf{H}_t$**，把「历史」压缩进去：

$$\mathbf{H}_t = \phi(\mathbf{X}_t \mathbf{W}_{xh} + \mathbf{H}_{t-1} \mathbf{W}_{hh} + \mathbf{b}_h), \qquad \mathbf{O}_t = \mathbf{H}_t \mathbf{W}_{hq} + \mathbf{b}_q$$

- **字符定义**：$\mathbf{X}_t$ 当前输入，$\mathbf{H}_t$ 当前隐状态，$\mathbf{H}_{t-1}$ 上一步隐状态，$\mathbf{W}_{xh},\mathbf{W}_{hh},\mathbf{W}_{hq}$ 是可学习权重，$\phi$ 是激活函数（如 tanh），$\mathbf{O}_t$ 是输出。
- **公式描述**：当前隐状态 = 当前输入 + 上一步隐状态 的融合。$\mathbf{W}_{hh}$ 是**关键**——它让信息沿时间步传递，这就是「记忆」。注意所有时间步**共享同一套权重**。
- **痛点**：训练用 BPTT（沿时间反向传播），梯度连乘容易**梯度消失/爆炸**，导致 RNN 记不住**长程依赖**。

### 251 GRU —— 用门控选择性记忆
GRU 用两个**门**来决定「记多少、忘多少」：**重置门 $\mathbf{R}_t$**（要不要忘掉过去）、**更新门 $\mathbf{Z}_t$**（用旧状态还是新状态）。

$$\mathbf{H}_t = \mathbf{Z}_t \odot \mathbf{H}_{t-1} + (1 - \mathbf{Z}_t) \odot \tilde{\mathbf{H}}_t$$

- **字符定义**：$\mathbf{Z}_t \in (0,1)$ 是更新门，$\tilde{\mathbf{H}}_t$ 是候选隐状态，$\odot$ 是逐元素相乘。
- **公式描述**：更新门像一个「混合旋钮」——$\mathbf{Z}_t$ 接近 1 就**保留旧记忆**（跳过当前输入，长期记忆得以保留），接近 0 就**采纳新信息**。门控让梯度能「抄近路」流过，缓解梯度消失。

### 252 LSTM —— 更精细的三门 + 记忆元
LSTM 比 GRU 多一个独立的**记忆元 $\mathbf{C}_t$**，用三个门控制：**遗忘门 $\mathbf{F}_t$**、**输入门 $\mathbf{I}_t$**、**输出门 $\mathbf{O}_t$**。

$$\mathbf{C}_t = \mathbf{F}_t \odot \mathbf{C}_{t-1} + \mathbf{I}_t \odot \tilde{\mathbf{C}}_t, \qquad \mathbf{H}_t = \mathbf{O}_t \odot \tanh(\mathbf{C}_t)$$

- **字符定义**：$\mathbf{C}_t$ 记忆元（长期记忆），$\mathbf{F}_t$ 遗忘门（保留多少旧记忆），$\mathbf{I}_t$ 输入门（写入多少新信息），$\mathbf{O}_t$ 输出门（从记忆元读出多少给隐状态），$\tilde{\mathbf{C}}_t$ 候选记忆。
- **公式描述**：记忆元 $\mathbf{C}_t$ 像一条「传送带」——遗忘门决定擦掉多少旧货，输入门决定放上多少新货；隐状态 $\mathbf{H}_t$ 再由输出门从传送带上「按需读取」。这种读写分离让 LSTM 能更稳地保持长期信息。

### 253 深层 RNN / 254 双向 RNN
- **深层 RNN**：把多个 RNN 层**纵向堆叠**，下层输出作上层输入，增强表达能力。
- **双向 RNN**：同时跑一个**前向**和一个**后向** RNN，拼接两者隐状态——这样每个位置能同时看到**过去和未来**的上下文。注意：双向 RNN **不能用于自回归预测**（因为预测时拿不到未来），主要用于特征抽取/填空类任务。

## 三、序列到序列（255–258）：处理「输入输出都是序列」

### 255 机器翻译数据集
翻译是典型的**序列对**任务（英文 → 法文，长度还不一样）。预处理要点：小写化、标点加空格、构建源/目标两个词表、用 `<bos>`(句首)/`<eos>`(句尾)/`<pad>`(填充) 特殊词元，把每句**截断或填充**到定长，并记录**有效长度 valid_len**。

### 256 编码器-解码器架构
通用范式：**编码器**把输入序列压成一个中间表示（上下文），**解码器**再从中生成输出序列。这是一个抽象接口，RNN、CNN、Transformer 都能往里套。

### 257 seq2seq —— 用 RNN 实现编码器-解码器
- 编码器用 RNN 读完源句，把**最后的隐状态**作为上下文向量 $\mathbf{c}$。
- 解码器用 RNN，以 $\mathbf{c}$ 为初始状态，逐词生成译文。
- 训练用 **teacher forcing**（喂真实标签作下一步输入）；评价用 **BLEU**。

$$\mathrm{BLEU} = \exp\!\left(\min\left(0,\, 1 - \frac{\mathrm{len}_{\text{label}}}{\mathrm{len}_{\text{pred}}}\right)\right) \prod_{n=1}^{k} p_n^{\,1/2^n}$$

- **字符定义**：$\mathrm{len}_{\text{label}}/\mathrm{len}_{\text{pred}}$ 是参考/预测译文长度，$p_n$ 是 $n$-gram 匹配精度，$k$ 是最大 gram 阶数。
- **公式描述**：前项是**短句惩罚**（译文太短扣分），后项是各阶 n-gram 精度乘积且**长匹配权重更大**。BLEU 越接近 1 越好。
- **暴露的痛点**：整句被压成**一个固定向量 $\mathbf{c}$**，长句信息装不下 → **信息瓶颈**。这正是注意力要解决的问题。

### 258 束搜索 —— 推理阶段怎么搜
训练有真实标签，但**推理时没有**，必须逐词搜索最优序列。三种策略是同一谱系：

| 束宽 $k$ | 方法 | 特点 |
|----------|------|------|
| $k=1$ | 贪心搜索 | 每步取最大概率，最快但易陷局部最优 |
| $1<k<n$ | **束搜索** | 每步保留最好的 $k$ 个候选，质量/算力平衡 |
| $k=n$ | 穷举搜索 | 最优但 $n^T$ 不可行 |

束搜索最终用**长度归一化**评分 $\frac{1}{L^\alpha}\sum \log p$（$\alpha\approx0.75$），避免偏好过短句子。

## 四、注意力 → Transformer（259–263）：抛弃循环，走向并行

### 259 注意力机制 —— query/key/value 的诞生
心理学动机：人通过**随意线索（query，主动意图）**和**不随意线索（key）**来选择关注点。注意力的统一写法：

$$f(x) = \sum_{i} \alpha(x, x_i)\, y_i$$

- **字符定义**：$x$ 是 query，$x_i$ 是 key，$y_i$ 是 value，$\alpha(x,x_i)$ 是注意力权重（和为 1）。
- **公式描述**：输出 = 对所有 value 的**加权平均**，权重由「query 和 key 的匹配度」决定。**不同注意力机制的区别全在 $\alpha$ 的设计。** Nadaraya-Watson 核回归是它最早的非参实例。

### 260 注意力分数 —— 权重到底怎么算
权重 = 对**注意力分数 $a(\mathbf{q},\mathbf{k})$** 做 softmax。两种主流分数函数：

$$\text{加性：}\; a(\mathbf{q},\mathbf{k}) = \mathbf{w}_v^\top \tanh(\mathbf{W}_q\mathbf{q} + \mathbf{W}_k\mathbf{k}) \qquad\qquad \text{缩放点积：}\; a(\mathbf{q},\mathbf{k}) = \frac{\mathbf{q}^\top \mathbf{k}}{\sqrt{d}}$$

- **字符定义**：$\mathbf{q},\mathbf{k}$ 查询/键向量，$\mathbf{W}_q,\mathbf{W}_k,\mathbf{w}_v$ 可学习参数，$d$ 是 q/k 维度。
- **公式描述**：**加性**用一个小 MLP 打分，允许 q/k 维度不同、但慢；**缩放点积**直接用点积、无参数、可纯矩阵乘法实现，最快。除 $\sqrt{d}$ 是为了**控制方差**，防止高维点积过大使 softmax 梯度消失。Transformer 选缩放点积。
- **masked softmax**：把无效（填充）位置分数设 $-\infty$，使其权重≈0，处理变长序列。

### 261 注意力 seq2seq —— Bahdanau 注意力打破瓶颈
不再用固定的 $\mathbf{c}$，而是**每个解码步动态算一个上下文向量**：

$$\mathbf{c}_{t'} = \sum_{t=1}^{T} \alpha(\mathbf{s}_{t'-1}, \mathbf{h}_t)\, \mathbf{h}_t$$

- **字符定义**：$\mathbf{s}_{t'-1}$ 解码器上一步隐状态（=**query**），$\mathbf{h}_t$ 编码器各步输出（=**key 和 value**），$\mathbf{c}_{t'}$ 第 $t'$ 步专属上下文。
- **公式描述**：解码每个词时，用「我下一步想译什么」(query) 去源句里**动态挑选**该关注的部分。这是 **query 与 key/value 来源不同**的**交叉注意力（cross-attention）**。

### 262 自注意力 —— 序列自己看自己
让序列里**每个元素同时当 query/key/value**：$\mathbf{y}_i = f(\mathbf{x}_i, (\mathbf{x}_1,\mathbf{x}_1),\dots,(\mathbf{x}_n,\mathbf{x}_n))$。

| 指标 | CNN | RNN | 自注意力 |
|------|-----|-----|----------|
| 计算复杂度 | $O(knd^2)$ | $O(nd^2)$ | $O(n^2d)$ |
| 并行度 | $O(n)$ | $O(1)$ ❌ | $O(n)$ ✅ |
| 最长路径 | $O(n/k)$ | $O(n)$ ❌ | $O(1)$ ✅ |

- **核心结论**：自注意力 = **完全并行 + 任意两词一步直达**（长依赖友好），代价是 $O(n^2)$ 的平方复杂度。
- **位置编码**：自注意力本身不感知顺序，用 sin/cos 矩阵 $\mathbf{P}$ 加到输入：$p_{i,2j}=\sin(i/10000^{2j/d})$，$p_{i,2j+1}=\cos(\cdots)$。它同时编码**绝对位置**（唯一指纹，类比二进制）和**相对位置**（位置 $i{+}\delta$ 可由 $i$ 经一个与 $i$ 无关的旋转矩阵线性得到）。

### 263 Transformer —— 纯注意力的集大成
把上面所有积木拼成**纯注意力的编码器-解码器**（无 RNN/CNN）。每个块 = 多头(自)注意力 + 基于位置的 FFN + 残差&层归一化。

**多头注意力**（用 $h$ 个独立子空间抽取不同信息）：

$$\mathbf{h}_i = f(\mathbf{W}_i^{(q)}\mathbf{q}, \mathbf{W}_i^{(k)}\mathbf{k}, \mathbf{W}_i^{(v)}\mathbf{v}), \qquad \text{输出} = \mathbf{W}_o[\mathbf{h}_1; \dots; \mathbf{h}_h]$$

- **字符定义**：$\mathbf{W}_i^{(q,k,v)}$ 第 $i$ 头的投影矩阵，$h$ 头数，$\mathbf{W}_o$ 合并矩阵。
- **公式描述**：每个头把 q/k/v 投到不同子空间各算一次注意力（学不同的关注模式，如短/长距离关系），再拼接经 $\mathbf{W}_o$ 融合。

**三种注意力分工**（务必记牢）：

| 位置 | 类型 | query 来源 | key/value 来源 |
|------|------|-----------|----------------|
| 编码器 | 自注意力 | 源序列 | 源序列 |
| 解码器第 1 层 | **带掩码**自注意力 | 目标序列(已生成) | 目标序列(已生成) |
| 解码器第 2 层 | 编码器-解码器注意力 | 目标序列 | **编码器输出** |

- **带掩码**：解码自回归，预测第 $i$ 词只能看前 $i{-}1$ 个，用掩码屏蔽未来（保证因果性）。
- **基于位置 FFN**：对每个位置独立做两层 MLP，等价两层 1×1 卷积，负责「逐位置加深非线性」。
- **层归一化（非批归一化）**：对每个样本自身归一化，适合变长 NLP；配残差连接（Add&Norm）支撑深层堆叠。

## 五、把整条主线连起来（最重要的一节）

### 5.1 一句话演进史

```
序列怎么建模？           → 自回归 / 语言模型      (246-248)
怎么有记忆？             → RNN 的隐状态           (249-250)
记不住长程怎么办？       → GRU / LSTM 的门控      (251-252)
表达力/上下文不够？      → 深层 / 双向 RNN        (253-254)
输入输出都是序列？       → 编码器-解码器 / seq2seq (255-257)
推理怎么搜好句子？       → 束搜索                 (258)
固定上下文向量瓶颈？     → 注意力 / Bahdanau      (259-261)
循环太慢、长依赖难？     → 自注意力 + 位置编码    (262)
干脆不要循环？           → Transformer           (263)
```

### 5.2 两条贯穿全程的主线

**主线 A：记忆 vs 并行的拉锯**
- RNN：用循环获得记忆，但被迫**串行**（并行度 $O(1)$）、长路径 $O(n)$ → 慢且长依赖难学。
- 注意力/Transformer：用「全连接式回看」换来**并行 $O(n)$ + 路径 $O(1)$** → 快且长依赖友好，代价是 $O(n^2)$ 计算。
- **这就是 Transformer 取代 RNN 的根本原因。**

**主线 B：query / key / value 的统一语言**
- 259 提出：注意力 = 用 query 匹配 key、对 value 加权平均。
- 260 明确：匹配度怎么算（加性 / 缩放点积）。
- 261 应用：query=解码器、key/value=编码器（**交叉**注意力）。
- 262 推广：q/k/v 同源（**自**注意力）。
- 263 集成：三种注意力 + 多头并行，构成 Transformer。

### 5.3 关键「补丁」清单（每个都对应一个具体问题）

| 补丁 | 解决什么 | 出现章节 |
|------|----------|----------|
| 门控（GRU/LSTM） | RNN 梯度消失、长依赖 | 251-252 |
| 注意力 | seq2seq 固定上下文瓶颈 | 259-261 |
| 位置编码 | 自注意力不感知顺序 | 262 |
| 多头 | 单一注意力只能看一种关系 | 263 |
| 掩码 | 解码器偷看未来 | 263 |
| 残差 + 层归一化 | 深层网络难训练 | 263 |
| 除 $\sqrt{d}$ | 高维点积 softmax 梯度消失 | 260/263 |
| 束搜索 + 长度归一化 | 推理搜索质量 / 偏好短句 | 258 |

### 5.4 自检问题（能答上来说明真懂了）
1. 为什么 RNN 难学长依赖？门控是怎么缓解的？
2. seq2seq 的「信息瓶颈」具体指什么？注意力如何打破它？
3. 交叉注意力和自注意力的区别？分别在 Transformer 哪里出现？
4. 为什么自注意力需要位置编码，而 RNN 不需要？
5. 缩放点积为什么要除 $\sqrt{d}$？多头注意力「多」在哪？
6. 解码器的掩码在解决什么问题？训练和推理时分别如何体现？
7. 为什么 Transformer 用层归一化而不是批量归一化？
8. 贪心、束搜索、穷举三者什么关系？束宽 $k$ 的意义？

## 六、结语

这 18 章不是 18 个孤立模型，而是**一条不断自我修正的进化链**：

> RNN 给了序列「记忆」，但记忆有限且只能串行；
> 门控（GRU/LSTM）让记忆更持久；
> seq2seq 让模型能「翻译」，但被固定上下文卡住喉咙；
> 注意力让模型学会「按需回看」，打破瓶颈；
> 自注意力把「回看」推广到序列内部，换来并行与短路径；
> Transformer 索性丢掉循环，纯靠注意力 + 位置编码 + 多头 + 残差，成为现代大模型的基石。

掌握了这条主线，再去看 BERT、GPT 等模型，你会发现它们都只是 Transformer 的「编码器版」或「解码器版」的放大——**底层语言，你已经全部学过了。**